In [1]:
!pip install pandas requests ipywidgets jupyterlab_widgets

     ---------------------------------------- 11.4/11.4 MB 1.5 MB/s eta 0:00:00
     ---------------------------------------- 64.7/64.7 KB 1.8 MB/s eta 0:00:00
     ------------------------------------ 139.8/139.8 KB 395.0 kB/s eta 0:00:00
     -------------------------------------- 914.9/914.9 KB 1.0 MB/s eta 0:00:00
     -------------------------------------- 348.5/348.5 KB 1.1 MB/s eta 0:00:00
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
     -------------------------------------- 71.0/71.0 KB 980.1 kB/s eta 0:00:00
     -------------------------------------- 107.2/107.2 KB 1.2 MB/s eta 0:00:00
     -------------------------------------- 131.6/131.6 KB 1.1 MB/s eta 0:00:00
     ---------------------------------------- 2.2/2.2 MB 1.2 MB/s eta 0:00:00
  Using cached ipython-8.18.1-py3-none-any.whl (808 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
     ---------------------------------------- 1.2/1.2 MB 1.1 MB/s eta 0:00:00
  Using cached decorator-5.2.1-py3

You should consider upgrading via the 'C:\Users\acer\paleofauna-model\landmask\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
import os
import pygplates
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from ipywidgets import IntSlider, VBox, Output
from IPython.display import display

from shapely.geometry import Polygon as ShapelyPolygon
import math

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

BASE_PATH = r"C:\Users\acer\paleofauna-model\GPlates 2.5.0\GeoData\FeatureCollections\AltPlateReconstructions\Muller_etal_2022"

ROTATION_FILE = os.path.join(BASE_PATH, "1000_0_rotfile.rot")
LANDMASK_FILE = os.path.join(
    BASE_PATH,
    "COB_polygons_and_coastlines_combined_1000_0_Merdith_etal.gpml"
)

rotation_model = pygplates.RotationModel(ROTATION_FILE)
landmask_features = pygplates.FeatureCollection(LANDMASK_FILE)

# --------------------------------------------------
# DATA LOADERS
# --------------------------------------------------

def get_plate_boundaries(time):
    if time > 410:
        paths = [
            os.path.join(BASE_PATH, '1000-410-Convergence.gpml'),
            os.path.join(BASE_PATH, '1000-410-Divergence.gpml'),
            os.path.join(BASE_PATH, '1000-410-Transforms.gpml')
        ]
    elif 250 < time <= 410:
        paths = [os.path.join(BASE_PATH, '410-250_plate_boundaries.gpml')]
    else:
        paths = [os.path.join(BASE_PATH, '250-0_plate_boundaries.gpml')]

    features = []
    for p in paths:
        features += pygplates.FeatureCollection(p)
    return features


def extract_land_polygons():
    polygons = []
    for feat in landmask_features:
        geom = feat.get_geometry()
        if geom and "Polygon" in geom.__class__.__name__:
            polygons.append(feat)
    return polygons


def reconstruct_features(features, time):
    reconstructed = []
    pygplates.reconstruct(features, rotation_model, reconstructed, time)
    return reconstructed


def reconstruct_coastlines(time):
    reconstructed = []
    pygplates.reconstruct(landmask_features, rotation_model, reconstructed, time)
    return reconstructed


def reconstruct_land_polygons(time):
    raw = extract_land_polygons()
    reconstructed = []
    pygplates.reconstruct(raw, rotation_model, reconstructed, time)
    return reconstructed


# --------------------------------------------------
# PLOTTING HELPERS
# --------------------------------------------------

def plot_reconstructed_features(ax, reconstructed, color, linewidth=0.6):
    for feature in reconstructed:
        geom = feature.get_reconstructed_geometry()
        if geom and hasattr(geom, 'to_lat_lon_list'):
            coords = geom.to_lat_lon_list()
            if coords:
                lats, lons = zip(*coords)
                ax.plot(
                    lons, lats,
                    transform=ccrs.Geodetic(),
                    color=color,
                    linewidth=linewidth,
                    zorder=10
                )


def geom_to_shapely_polygon(geom):
    coords = geom.to_lat_lon_list()
    if not coords or len(coords) < 4:
        return None

    lats, lons = zip(*coords)
    poly = ShapelyPolygon(zip(lons, lats))

    # must be topologically valid in the plane
    if not poly.is_valid:
        return None

    # crude planar sanity limits to kill obvious explosions
    if poly.area > 20000:
        return None

    if poly.area < 1.0:
        return None

    # reject polygons that almost span the whole longitude range
    minx, miny, maxx, maxy = poly.bounds
    if (maxx - minx) > 270:
        return None

    return poly


def render_landmask(ax, time):
    reconstructed = reconstruct_land_polygons(time)

    geoms = []
    skipped = 0
    flipped = 0

    for feature in reconstructed:
        geom = feature.get_reconstructed_geometry()
        if geom is None:
            skipped += 1
            continue

        # -------------------------------
        # ORIENTATION FIX (SPHERICAL)
        # -------------------------------
        # pygplates area is in steradians on the unit sphere
        spherical_area = geom.get_area()

        # if polygon covers more than half the globe, flip it
        if spherical_area > 2 * math.pi:
            geom = geom.get_reversed()
            flipped += 1

        # now convert to planar shapely for drawing
        shp = geom_to_shapely_polygon(geom)
        if shp is None:
            skipped += 1
            continue

        geoms.append(shp)

    ax.add_geometries(
        geoms,
        crs=ccrs.PlateCarree(),   # lon/lat on the sphere
        facecolor="lightgreen",
        edgecolor="none",
        zorder=5
    )

    print(f"[LANDMASK] Rendered {len(geoms)}, flipped {flipped}, skipped {skipped}")


# --------------------------------------------------
# UI
# --------------------------------------------------

out = Output()

slider = IntSlider(
    value=70,
    min=0,
    max=1000,
    step=5,
    description="Time (Ma)",
    continuous_update=False
)

def update_plot(change):
    with out:
        out.clear_output(wait=True)
        time = change["new"]

        fig = plt.figure(figsize=(12, 6))
        ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
        ax.set_global()
        ax.set_title(f"Landmask + Boundaries @ {time} Ma")

        # Plate boundaries
        boundaries = get_plate_boundaries(time)
        reconstructed_boundaries = reconstruct_features(boundaries, time)
        plot_reconstructed_features(ax, reconstructed_boundaries, color="blue")

        # Coastlines
        reconstructed_coasts = reconstruct_coastlines(time)
        plot_reconstructed_features(ax, reconstructed_coasts, color="saddlebrown", linewidth=0.4)

        # Landmask
        render_landmask(ax, time)

        plt.show()

slider.observe(update_plot, names="value")

display(VBox([slider, out]))
update_plot({"new": slider.value})